In [1]:
!pip install textblob nltk transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.7 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.

In [11]:
import gradio as gr
import difflib
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from textblob import TextBlob

class AutocorrectModel:
    def __init__(self, mode="low"):
        self.mode = mode.lower()
        if self.mode == "low":
            self.TextBlob = TextBlob
            print("Initialized Low-end mode (TextBlob).")
        elif self.mode == "high":
            print("Initializing High-end mode (Transformers)... This will take a moment.")
            model_name = "vennify/t5-base-grammar-correction"
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name, tie_word_embeddings=False)
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
            self.model = self.model.to(self.device)
            print(f"High-end mode initialized on {self.device.upper()}.")

    def correct(self, text, num_return_sequences=1):
        if self.mode == "low":
            corrected = str(self.TextBlob(text).correct())
            return [corrected] if num_return_sequences > 1 else corrected
            
        elif self.mode == "high":
            input_text = "grammar: " + text
            inputs = self.tokenizer(input_text, return_tensors="pt").to(self.device)
            outputs = self.model.generate(
                **inputs, 
                max_length=128,
                num_return_sequences=num_return_sequences,
                num_beams=max(5, num_return_sequences),
                early_stopping=True
            )
            results = [self.tokenizer.decode(out, skip_special_tokens=True) for out in outputs]
            
            unique_results = []
            for r in results:
                if r not in unique_results:
                    unique_results.append(r)
            
            if num_return_sequences > 1:
                return unique_results
            else:
                return unique_results[0] if unique_results else text

# Initialize Models
print("Loading models...")
low_model = AutocorrectModel(mode="low")
high_model = AutocorrectModel(mode="high")
print("Models ready!")

def get_diff_html(original, corrected):
    matcher = difflib.SequenceMatcher(None, original.split(), corrected.split())
    html = []
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == 'equal':
            html.append(" ".join(original.split()[i1:i2]))
        elif op == 'delete':
            html.append(f'<span style="background-color: #ffcccc; color: #cc0000; text-decoration: line-through; padding: 0 2px; border-radius: 2px;">{" ".join(original.split()[i1:i2])}</span>')
        elif op == 'insert':
            html.append(f'<span style="background-color: #ccffcc; color: #008800; font-weight: bold; padding: 0 2px; border-radius: 2px;">{" ".join(corrected.split()[j1:j2])}</span>')
        elif op == 'replace':
            html.append(f'<span style="background-color: #ffcccc; color: #cc0000; text-decoration: line-through; padding: 0 2px; border-radius: 2px;">{" ".join(original.split()[i1:i2])}</span>')
            html.append(f'<span style="background-color: #ccffcc; color: #008800; font-weight: bold; padding: 0 2px; border-radius: 2px;">{" ".join(corrected.split()[j1:j2])}</span>')
    return f'<div style="font-family: sans-serif; font-size: 1.1em; padding: 10px; border: 1px solid #ddd; border-radius: 5px; background: #fff;">{" ".join(html)}</div>'

def correct_single(text, mode, num_suggestions):
    if not text.strip(): return "", "", ""
    model = high_model if "High" in mode else low_model
    results = model.correct(text, num_return_sequences=int(num_suggestions))
    primary = results[0] if isinstance(results, list) else results
    alternatives = "\n".join([f"{i+1}. {r}" for i, r in enumerate(results[1:])]) if isinstance(results, list) and len(results) > 1 else "No other suggestions found."
    return get_diff_html(text, primary), primary, alternatives

def correct_file(file_obj, mode):
    if file_obj is None: return None
    model = high_model if "High" in mode else low_model
    with open(file_obj.name, 'r', encoding='utf-8') as f: lines = f.readlines()
    corrected_lines = []
    for line in lines:
        if not line.strip():
            corrected_lines.append("\n")
            continue
        res = model.correct(line.strip(), num_return_sequences=1)
        corrected_lines.append((res[0] if isinstance(res, list) else res) + "\n")
    out_file = "corrected_document.txt"
    with open(out_file, 'w', encoding='utf-8') as f: f.writelines(corrected_lines)
    return out_file

# Build Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ✍️ AI Autocorrect Pro")
    gr.Markdown("Experience next-gen spelling and grammar correction.")
    
    with gr.Tabs():
        with gr.TabItem("Interactive Correction"):
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label="Original Text", placeholder="Type your sentence here...", lines=4)
                    mode_selector = gr.Radio(choices=["Low-End (TextBlob)", "High-End (Transformers)"], value="High-End (Transformers)", label="Correction Mode")
                    num_sug = gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Number of Alternative Suggestions")
                    submit_btn = gr.Button("Correct Text", variant="primary")
                with gr.Column():
                    gr.Markdown("### Visual Diff")
                    diff_output = gr.HTML(value="<i>Type text and click 'Correct Text' to see changes...</i>")
                    primary_output = gr.Textbox(label="Primary Correction", lines=2, interactive=False)
                    alt_output = gr.Textbox(label="Alternative Suggestions", lines=3, interactive=False)
            submit_btn.click(fn=correct_single, inputs=[input_text, mode_selector, num_sug], outputs=[diff_output, primary_output, alt_output])
            
        with gr.TabItem("Batch Document Processing"):
            with gr.Row():
                with gr.Column():
                    file_input = gr.File(label="Upload .txt File", file_types=[".txt"])
                    batch_mode_selector = gr.Radio(choices=["Low-End (TextBlob)", "High-End (Transformers)"], value="High-End (Transformers)", label="Correction Mode")
                    process_btn = gr.Button("Process Document", variant="primary")
                with gr.Column():
                    file_output = gr.File(label="Download Corrected File")
            process_btn.click(fn=correct_file, inputs=[file_input, batch_mode_selector], outputs=file_output)

demo.launch(share=True)


Loading models...
Initialized Low-end mode (TextBlob).
Initializing High-end mode (Transformers)... This will take a moment.


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

High-end mode initialized on CUDA.
Models ready!


/tmp/ipykernel_59/773219471.py:94: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://9e37e7bbfde0c88534.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
